# Agentic VQA Pipeline su Kaggle con due T4

Questo notebook esegue DOTS.OCR e GLiNER sulla prima GPU e il VLM Ollama sulla seconda GPU. E' pensato per il repository `Agentic-VQA-Pipeline` e mantiene il checkpoint dei risultati in `/kaggle/working`.

Prima di eseguirlo, in **Notebook options** abilita:

- Accelerator: **GPU T4 x2**
- Internet: **On** (necessario per installare Ollama e scaricare i modelli)

Aggiungi inoltre come Kaggle Dataset sia il repository sia i dati DUDE, oppure indica un repository Git pubblico nei parametri qui sotto.

In [ ]:
# ---- PARAMETRI DA ADATTARE ----
PROJECT_SOURCE = ""  # Esempio: /kaggle/input/agentic-vqa-pipeline
GIT_REPOSITORY_URL = ""  # Alternativa: URL Git pubblico del repository
GIT_REF = "main"

INPUT_JSON_PATH = "/kaggle/input/datasets/matteopetrelli/dude-mixed/DUDE_mixed_test.json"
IMAGE_DIR = "/kaggle/input/datasets/matteopetrelli/dude-train/content/DUDE_train-val-test_binaries/images/train"
DOCUMENT_PATH = ""  # PDF, DOCX, TXT, immagine o cartella di immagini per la prova singola
CUSTOM_QUESTION = "What information does this document provide?"
OUTPUT_JSON_PATH = "/kaggle/working/unanswerability_diagnostic_results_gemma3.json"

OLLAMA_MODEL = "gemma3:4b"
EVIDENCE_GPU = 0
VLM_GPU = 1
ALLOW_SINGLE_GPU_FALLBACK = True
SAMPLING_PERCENTAGE = 0.1
RUN_SMOKE_TEST = True
RUN_FULL_EXPERIMENT = False
MAX_DOCUMENT_MB = 100
CHUNK_SIZE = 1800
CHUNK_OVERLAP = 200

## 1. Configurazione dell'ambiente Kaggle

Individua il progetto, installa le dipendenze e configura `HF_TOKEN` dai Kaggle Secrets.

In [ ]:
from pathlib import Path

WORKING_DIR = Path("/kaggle/working") if Path("/kaggle").exists() else Path.cwd()

# Find or clone the project
import os, sys
# Minimal bootstrap: find project before importing kaggle_utils
_clone_dir = WORKING_DIR / "Agentic-VQA-Pipeline"
for _candidate in ([Path(PROJECT_SOURCE).expanduser()] if PROJECT_SOURCE else []) + [_clone_dir, Path.cwd()]:
    if _candidate and (_candidate / "kaggle_utils.py").is_file():
        if str(_candidate) not in sys.path:
            sys.path.insert(0, str(_candidate))
        break
else:
    if GIT_REPOSITORY_URL:
        import subprocess
        subprocess.run(["git", "clone", "--depth", "1", "--branch", GIT_REF, GIT_REPOSITORY_URL, str(_clone_dir)], check=True)
        sys.path.insert(0, str(_clone_dir))

from kaggle_utils import find_project, setup_environment

PROJECT_DIR = find_project(PROJECT_SOURCE, GIT_REPOSITORY_URL, GIT_REF, WORKING_DIR)
setup_environment(PROJECT_DIR)

## 2. Rilevamento e assegnazione delle GPU

PyTorch vede entrambe le GPU. DOTS/GLiNER useranno `cuda:0`; Ollama ricevera' solo GPU 1.

In [ ]:
from kaggle_utils import detect_gpus

EVIDENCE_DEVICE, VLM_GPU = detect_gpus(EVIDENCE_GPU, VLM_GPU, ALLOW_SINGLE_GPU_FALLBACK)

## 3. Caricamento del documento

Per una prova libera, imposta `DOCUMENT_PATH` a un PDF, DOCX, TXT, immagine o cartella. Se resta vuoto, lo smoke test usera' il primo elemento del dataset.

In [ ]:
from kaggle_utils import prepare_document

CUSTOM_IMAGE_PATHS = prepare_document(DOCUMENT_PATH, WORKING_DIR, MAX_DOCUMENT_MB)
print(f"Pagine preparate: {len(CUSTOM_IMAGE_PATHS)}")

## 4. Elaborazione con DOTS e anteprima

Carica DOTS e GLiNER sulla GPU di elaborazione, poi esegue un'anteprima su una pagina.

In [ ]:
import json
import config
from kaggle_utils import clean_text, chunk_by_characters, chunk_by_sentences, chunk_by_paragraphs, chunk_by_tokens

config.INPUT_JSON_PATH = INPUT_JSON_PATH
config.IMAGE_DIR = IMAGE_DIR
config.OUTPUT_JSON_PATH = OUTPUT_JSON_PATH
config.OLLAMA_VLM = OLLAMA_MODEL
config.EVIDENCE_DEVICE = EVIDENCE_DEVICE
config.SAMPLING_PERCENTAGE = SAMPLING_PERCENTAGE
config.PROMPT_PROFILE = "document_focused"

from diagnostic_agent.engine import DocumentEngine
from run_experiments import _image_paths

if CUSTOM_IMAGE_PATHS:
    SMOKE_QUESTION = CUSTOM_QUESTION
    SMOKE_IMAGE_PATHS = CUSTOM_IMAGE_PATHS
else:
    input_file = Path(INPUT_JSON_PATH)
    if not input_file.is_file():
        raise FileNotFoundError(f"Dataset not found: {input_file}")
    dataset = json.loads(input_file.read_text(encoding="utf-8"))
    questions = dataset.get("corrupted_questions", [])
    if not questions:
        raise ValueError("Dataset contains no corrupted_questions")
    SMOKE_ITEM = questions[0]
    SMOKE_QUESTION = SMOKE_ITEM.get("corrupted_question", "")
    SMOKE_IMAGE_PATHS = _image_paths(SMOKE_ITEM)

if not SMOKE_IMAGE_PATHS or not all(Path(p).is_file() for p in SMOKE_IMAGE_PATHS):
    raise FileNotFoundError("One or more document pages are not available")

engine = DocumentEngine()
PREVIEW_LAYOUT = engine.get_layout(SMOKE_IMAGE_PATHS[0])
PREVIEW_OCR = clean_text("\n".join(str(block.get("text_content") or "") for block in PREVIEW_LAYOUT))
PREVIEW_TAGGED_OCR, PREVIEW_ENTITIES = engine.tag_text_with_gliner(PREVIEW_OCR)

tokenizer = getattr(engine.dots_processor, "tokenizer", None)
chunk_counts = {
    "characters": len(chunk_by_characters(PREVIEW_OCR, CHUNK_SIZE, CHUNK_OVERLAP)),
    "sentences": len(chunk_by_sentences(PREVIEW_OCR, CHUNK_SIZE)),
    "paragraphs": len(chunk_by_paragraphs(PREVIEW_OCR, CHUNK_SIZE)),
}
if tokenizer is not None:
    chunk_counts["tokens"] = len(chunk_by_tokens(PREVIEW_OCR, tokenizer))

print(f"Question: {SMOKE_QUESTION}")
print(f"Pages: {len(SMOKE_IMAGE_PATHS)} | DOTS blocks (page 1): {len(PREVIEW_LAYOUT)}")
print(f"GLiNER entities (page 1): {len(PREVIEW_ENTITIES)}")
print(f"Chunk comparison: {chunk_counts}")

## 5. Anteprima del prompt

Costruisce un'anteprima del prompt con le evidenze estratte da GPU 0.

In [ ]:
import torch
from diagnostic_agent.prompts.catalog import get_prompt

preview_blocks = []
for block in PREVIEW_LAYOUT:
    category = str(block.get("category") or "Text")
    text = clean_text(str(block.get("text_content") or ""))
    if text:
        preview_blocks.append(f"[{category}]: {text}")

preview_context = {
    "mode": "answer",
    "question": SMOKE_QUESTION,
    "image_paths": SMOKE_IMAGE_PATHS[:1],
    "structured_ocr": "\n".join(preview_blocks),
    "question_analysis": {},
    "diagnostic_results": [],
    "document_elements": sorted({str(block.get("category") or "Text") for block in PREVIEW_LAYOUT}),
    "quadrants": [],
}
PREVIEW_PROMPT = get_prompt("docel_cot_v4").builder(preview_context)
prompt_tokens = len(tokenizer.encode(PREVIEW_PROMPT, add_special_tokens=False)) if tokenizer else None
memory_gb = torch.cuda.memory_allocated(int(EVIDENCE_DEVICE.split(":")[1])) / 1024**3
print(f"Prompt preview: {len(PREVIEW_PROMPT):,} chars, {prompt_tokens or 'n/a'} tokens")
print(f"VRAM allocated on {EVIDENCE_DEVICE}: {memory_gb:.2f} GiB")
print(PREVIEW_PROMPT[:1200])

## 6. Caricamento del modello VLM (Ollama)

Avvia Ollama sulla GPU dedicata, scarica il modello e lo tiene residente.

In [ ]:
from kaggle_utils import start_ollama, stop_ollama

api_url = start_ollama(VLM_GPU, OLLAMA_MODEL, WORKING_DIR)
config.OLLAMA_URL = api_url
config.OLLAMA_VLM = OLLAMA_MODEL

## 7. Smoke test

Esegue una domanda completa. In caso di OOM libera la cache e suggerisce i parametri da ridurre.

In [ ]:
import subprocess
from agentic_pipeline import AgenticPipeline

pipeline = AgenticPipeline(engine, model_name=OLLAMA_MODEL)
SMOKE_RESULT = None

if RUN_SMOKE_TEST:
    try:
        SMOKE_RESULT = pipeline.process_question(SMOKE_QUESTION, SMOKE_IMAGE_PATHS)
    except torch.cuda.OutOfMemoryError as error:
        torch.cuda.empty_cache()
        raise RuntimeError(
            "VRAM exhausted during smoke test. Reduce pages or disable USE_4BIT_DOTS."
        ) from error
    print(json.dumps(SMOKE_RESULT, indent=2, ensure_ascii=False))

subprocess.run(
    ["nvidia-smi", "--query-gpu=index,name,memory.used,memory.free", "--format=csv"],
    check=True,
)

## 8. Run completo (con checkpointing e sampling)

Il run completo usa `run_experiments.main()` che include:
- **Checkpointing**: salva il risultato dopo ogni domanda, riprende da dove si era fermato
- **Sampling**: rispetta `SAMPLING_PERCENTAGE` per limitare le domande elaborate

Imposta `RUN_FULL_EXPERIMENT = True` dopo lo smoke test.

In [ ]:
if RUN_FULL_EXPERIMENT:
    if CUSTOM_IMAGE_PATHS:
        raise ValueError("Full run uses the dataset: leave DOCUMENT_PATH empty.")
    from run_experiments import main as run_full_experiment

    run_full_experiment(model_name=OLLAMA_MODEL, engine=engine)
else:
    print("Full run not executed. Set RUN_FULL_EXPERIMENT = True after the smoke test.")

## 9. Salvataggio dei risultati

Esporta risultati in JSON, CSV e TXT.

In [ ]:
from kaggle_utils import export_results

export_results(
    OUTPUT_JSON_PATH,
    smoke_result=SMOKE_RESULT,
    smoke_question=SMOKE_QUESTION,
    smoke_image_paths=SMOKE_IMAGE_PATHS,
    run_full=RUN_FULL_EXPERIMENT,
    working_dir=WORKING_DIR,
)

### Arresto del server

Esegui questa cella solo quando hai terminato. Il server viene comunque arrestato automaticamente alla chiusura del kernel.

In [ ]:
stop_ollama()
print("Server Ollama arrestato.")